# Goal 4 -- Autonomous Compliance Sentinel Agent
### SRH Heidelberg | Data Ethics and Responsible AI
**Authors:** Vikrant Singh and Kay Marc Muller

---

## What is this notebook?

This notebook builds the **Goal 4 Agent**, the final and most important piece of the Autonomous Compliance Sentinel project.

The system reads an internal project proposal and automatically checks whether it violates any of our 9 Responsible AI policies (RAI-01 through RAI-09).

---

## The full pipeline at a glance

```
Project Proposal (text)
        |
        v
[Signal 1] TF-IDF + Logistic Regression Classifier
        |
        | -- Compliant? --> STOP. No further action needed.
        |
        | -- Red Flag? --> continue to Signal 2
        v
[Signal 2] LLM Agent + Tools (RAG, XAI, Severity, Audit)
        |
        v
Human-in-the-Loop Gate (High severity = human must approve)
        |
        v
Final Decision + Audit Log
```

---

## Two signals, not one

| Signal | What it is | When it runs |
|--------|-----------|--------------|
| Signal 1 | TF-IDF + Logistic Regression, threshold=0.4 | Always, on every proposal |
| Signal 2 | LLM + 5 tools (RAG, XAI, severity, audit, verdict) | Only when Signal 1 flags Red Flag |

Signal 1 is fast and cheap. Signal 2 is slower but smarter. Together they are more reliable than either alone.

---

## The 9 RAI Policies

| Policy | Name | Severity |
|--------|------|----------|
| RAI-01 | Data Protection | High |
| RAI-02 | Transparency | High |
| RAI-03 | Fairness | High |
| RAI-04 | Human Dignity and Vulnerable Groups | High |
| RAI-05 | Prohibited Purpose | High |
| RAI-06 | Security | Medium |
| RAI-07 | Human Oversight | Medium |
| RAI-08 | Data Minimisation | Medium |
| RAI-09 | Explainability | Medium |

High severity violations are routed to a human reviewer before any action is taken.
Medium severity violations proceed automatically with an optional spot check.

---
## Step 0 -- Install dependencies

Run this cell once when setting up a new environment. It installs every library this notebook needs.

In [47]:
%pip install langchain langgraph langchain-community langchain-chroma langchain-huggingface langchain-groq sentence-transformers pdfplumber

Note: you may need to restart the kernel to use updated packages.


---
## Step 1 -- Imports

We group imports by what they do so the notebook is easy to follow.

**Group 1 -- Signal 1 (the ML classifier)**
We need `joblib` to reload the already-trained pipeline from disk.
We import `sklearn` pieces and `spacy` because `joblib` needs to find the class definitions
(`ThresholdAdjustor`, `token_pos`) in memory before it can unpack the saved file.
We do NOT retrain the model here. The model was trained in Goal 2 and saved as a `.pkl` file.

**Group 2 -- PDF loading and chunking**
`PDFPlumberLoader` reads the policy catalogue PDF page by page.
`RecursiveCharacterTextSplitter` cuts those pages into smaller chunks that fit inside a vector search.

**Group 3 -- Embedding and vector storage**
`HuggingFaceEmbeddings` turns text chunks into numbers (vectors) that can be searched by meaning.
`Chroma` stores those vectors on disk so we do not have to re-embed every time.

**Group 4 -- Tools and LLM**
`tool` is a decorator that turns a plain Python function into something an LLM can call by name.
`ChatGroq` connects to the Groq API to run `llama-3.1-8b-instant`, a free model that handles tool calling reliably.

**Group 5 -- Agent (LangGraph)**
`StateGraph`, `START`, `END` build the graph structure.
`ToolNode` is a prebuilt node that runs whichever tool the LLM requested.
`MemorySaver` keeps the graph state in memory so the human approval gate can pause and resume.
`HumanMessage`, `SystemMessage` are message types the LLM understands.
`TypedDict`, `Annotated`, `operator` define the shared memory structure of the graph.

In [48]:
# Group 1 -- Signal 1: reloading the already-trained classifier
import joblib
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
import spacy

# Group 2 -- PDF loading and chunking
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Group 3 -- Embedding and vector storage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Group 4 -- Tools and LLM
from langchain_core.tools import tool
from langchain_groq import ChatGroq

# Group 5 -- Agent (LangGraph)
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from typing import TypedDict, Annotated
import operator

print("All imports OK")

All imports OK


---
## Step 2a -- Reload Signal 1 (the trained classifier)

### What is Signal 1?

Signal 1 is the TF-IDF + Logistic Regression pipeline we trained in Goal 2.
It was saved to disk as `logRegpipelineV2.pkl` using `joblib`.

We do NOT retrain it here. We just load it back from disk, ready to use.

### Why do we import from xai2.py first?

When `joblib` saved the pipeline, it did not save the *code* of `ThresholdAdjustor` or `token_pos`.
It saved a *pointer* saying "find a class with this name in memory when loading."
If those names do not exist in memory, `joblib.load()` will crash with `AttributeError`.

Importing from `xai2.py` puts those names into memory so `joblib` can find them.

### What is ThresholdAdjustor?

It is a custom wrapper around Logistic Regression that applies a custom decision threshold.
The default sklearn threshold is 0.5: "classify as Red Flag if probability > 0.5."
Our threshold is 0.4: "classify as Red Flag if probability > 0.4."
This lowers the bar for flagging, which means fewer real violations get missed (higher recall).
Missing a real violation is worse than a false alarm in a compliance system.

In [49]:
from xai2 import ThresholdAdjustor, token_pos, nlp, explain_prediction, audit_trail

classifier_pipeline = joblib.load("logRegpipelineV2.pkl")
print("Signal 1 loaded.")
print("Pipeline steps:", list(classifier_pipeline.named_steps.keys()))

Signal 1 loaded.
Pipeline steps: ['tfidf', 'clf']


---
## Step 2b -- Load Signal 2 (the RAI Policy Catalogue PDF)

### What is Signal 2?

Signal 2 is the LLM agent. It only runs when Signal 1 flags a proposal as Red Flag.
Its job is to reason about *which* policy was violated and *why*, using tools.

The RAI Policy Catalogue is the knowledge base Signal 2 reasons over.
It contains 9 policies (RAI-01 through RAI-09), each with a Requirement, Violation Patterns, and a Compliant Example.

### What does PDFPlumberLoader do?

It reads the PDF page by page and returns a Python list of `Document` objects.
Each `Document` has two parts:
- `.page_content` -- the text on that page
- `.metadata` -- a dictionary containing the page number and file path

One `Document` per page. A 9-page PDF gives 9 `Document` objects.

In [50]:
path = "../../data/RAI_Policy_Catalogue.pdf"

loader = PDFPlumberLoader(path)
raw_pages = loader.load()

print(f"Pages loaded: {len(raw_pages)}")
print("First 300 characters of page 1:")
print(raw_pages[0].page_content[:300])

Pages loaded: 10
First 300 characters of page 1:
Autonomous Compliance Sentinel
Responsible AI Policy Catalogue, RAI-01 through RAI-09
SRH Heidelberg, Data Ethics and Responsible AI module. Author: Vikrant Singh and Kay Marc
Muller. This catalogue is the retrieval corpus for the Goal 4 compliance agent. Each policy below is
written in the same fix


---
## Step 3 -- Chunking

### Why do we chunk?

A full page of policy text is too long to match precisely in a vector search.
When a proposal mentions "chatbot without disclosure," we want to retrieve the specific
paragraph about chatbot transparency, not the entire page.
Chunking cuts each page into smaller, more targeted pieces.

### What is RecursiveCharacterTextSplitter?

It tries to split text at the biggest natural break it can find, in this priority order:
1. Blank lines (paragraph breaks) -- preferred, keeps ideas together
2. Single newlines
3. Sentence endings (". ")
4. Spaces between words
5. Mid-word, as an absolute last resort

This is format-agnostic: it does not rely on any specific heading style or author convention.

### Our settings: chunk_size=600, chunk_overlap=80

- `chunk_size=600` -- each chunk targets roughly 600 characters (about 4-6 sentences)
- `chunk_overlap=80` -- each chunk repeats the last 80 characters of the previous chunk

The overlap stops a sentence from being cut in half at a boundary with the second half lost.

### Known tradeoff

600 characters is smaller than some policy sub-sections (Requirement, Violation Patterns).
This means some sub-sections may be split across two chunks.
This slightly reduces retrieval precision but was chosen as a deliberate tradeoff.
It can be improved later by increasing chunk_size to 800.

In [51]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(raw_pages)
print(f"Total chunks created: {len(chunks)}")
print("\nFirst chunk text:")
print(chunks[0].page_content)

Total chunks created: 33

First chunk text:
Autonomous Compliance Sentinel
Responsible AI Policy Catalogue, RAI-01 through RAI-09
SRH Heidelberg, Data Ethics and Responsible AI module. Author: Vikrant Singh and Kay Marc
Muller. This catalogue is the retrieval corpus for the Goal 4 compliance agent. Each policy below is
written in the same fixed order every time, Policy ID and name, Severity, Regulatory Basis,
Requirement, Violation Patterns, and a Compliant Example, so that the meaning of any retrieved
chunk stays clear even without the rest of the document around it. Severity is a fixed lookup value,


---
## Step 4 -- Embedding

### What is an embedding?

An embedding is a list of numbers that represents the *meaning* of a piece of text.
Two chunks that mean similar things (both about "AI disclosure to users") will have
numerically similar vectors even if they use different words.
This is what allows semantic search: finding relevant chunks by meaning, not just keyword matching.

### BAAI/bge-small-en-v1.5

This is a small, fast embedding model from HuggingFace.
"Small" means it runs on CPU without a GPU.
"bge" stands for BAAI General Embedding.
It produces a 384-dimensional vector for any piece of text.

### model_kwargs={"device": "cpu"}

Explicitly tells the model to run on CPU, avoiding any ambiguity on machines without a GPU.

### encode_kwargs={"normalize_embeddings": True}

Scales every output vector to length 1 (unit length).
Chroma uses cosine similarity for search, which measures the angle between vectors.
Normalising removes vector length as a factor so similarity is judged purely on direction (meaning).

In [52]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
print("Embedding model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.


---
## Step 5 -- Vector Database (Chroma)

### What is a vector database?

A vector database stores text chunks as vectors and lets you search them by meaning.
When we search for "chatbot not disclosing AI involvement," Chroma converts that query
into a vector and returns the chunks whose vectors are closest to it.

### Why a separate collection "rai_policies_v2"?

We already have a collection called "rai_policies" from earlier work (one chunk per page).
This new collection uses the RecursiveCharacterTextSplitter (smaller, more precise chunks).
Using a different collection name keeps the two separate so they do not interfere.

### persist_directory

Without this, Chroma only exists in memory and disappears when the kernel restarts.
With it, the vectors are saved to disk and can be reloaded without re-embedding.

### IMPORTANT: delete_collection() before rebuilding

If you run this cell more than once without deleting first, Chroma will APPEND the chunks
again on top of the existing ones, creating duplicates.
Always delete the collection first if you need to rebuild it.

In [53]:
# If you are running this for the first time, just run Chroma.from_documents directly.
# If you have run it before and want to rebuild cleanly, uncomment the two lines below first:
# vector_db = Chroma(collection_name="rai_policies_v2", embedding_function=embeddings, persist_directory="Chroma")
# vector_db.delete_collection()

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rai_policies_v2",
    persist_directory="Chroma"
)

count = vector_db._collection.count()
print(f"Chunks stored in Chroma: {count}")

Chunks stored in Chroma: 297


---
## Step 6 -- Tools

### What is a tool in the context of an LLM agent?

A tool is a Python function that the LLM can call by name during reasoning.
The LLM does not run the code itself. It decides "I need to call this function with these arguments"
and LangGraph executes it and sends the result back to the LLM.

### What is the @tool decorator?

The `@tool` decorator turns a plain Python function into a LangChain `Tool` object.
It reads the function's **docstring** to know what the tool does and when to use it.
It reads the **type hints** (`text:str`, `k:int`) to know what arguments to send.
This is why the docstring is critical: it is the only description the LLM gets.

### Why do tools return strings?

Tools must return simple text so the LLM can read and reason about the result.
They cannot return DataFrames or Python objects directly.

### Our 5 tools

| Tool | What it does |
|------|-------------|
| `search_policies` | RAG: find relevant policy chunks from the catalogue |
| `get_trigger_words` | XAI: find which words in this proposal drove the classifier |
| `get_policy_severity` | Lookup: is this policy High or Medium severity? |
| `log_decision` | Audit: write the Article 12 audit trail entry |
| `write_compliance_verdict` | Verdict: write the final structured compliance finding |

### Tool 1 -- search_policies (RAG Retrieval)

This tool is the RAG (Retrieval Augmented Generation) component of the agent.

RAG means: instead of asking the LLM to remember policy text from training,
we retrieve the relevant text at runtime and give it to the LLM fresh.
This is more accurate and more auditable than relying on LLM memory.

The LLM calls this tool with a short query based on the trigger words it found.
The tool searches the Chroma vector database and returns the most relevant policy chunks.

**Deduplication with `seen`:**
The same chunk can appear multiple times in search results when chunk_size is small.
We use a Python `set` called `seen` to track which chunks we have already added.
Before adding a chunk, we check its first 100 characters (the "fingerprint") against `seen`.
If it is already there, we skip it with `continue`.

In [54]:
@tool
def search_policies(query:str, k:int=3)->str:
    """
    Search the RAI policy catalogue for clauses relevant to a query.
    Use this to find which of the 9 RAI policies apply to a proposal.
    Args:
        query: a short description of the concern or topic to search for
        k: number of matching policy chunks to return
    Returns:
        the matched policy text chunks, each labelled with its source page
    """
    results = vector_db.similarity_search(query, k=k)

    output = ""
    seen = set()
    for doc in results:
        page = doc.metadata.get("page", "unknown")
        fingerprint = doc.page_content[:100].strip()
        if fingerprint in seen:
            continue
        seen.add(fingerprint)
        output += f"[page {page}]\n{doc.page_content}\n\n"
    return output

# Sanity check
print(search_policies.invoke({"query": "chatbot not disclosing it is an AI", "k": 2}))

[page 2]
policy corresponds to the Transparency pillar.
Requirement
Any proposal in which an AI system interacts directly with a person, or makes or materially assists a
decision about a person, must describe how that person will be informed of the AI system's
involvement. This applies regardless of whether the interaction is a chatbot conversation, an
automated email, or a decision such as an approval or rejection. The disclosure must be part of the
system's design, not something left for a later stage of the project.
Violation Patterns




### Tool 2 -- get_trigger_words (XAI Grounding)

This tool exposes the explainability layer (Goal 3) as something the LLM can call itself.

**What are trigger words?**
The Logistic Regression model assigns a weight to every word in the vocabulary.
Positive weight = that word pushes the prediction toward Red Flag.
Negative weight = that word pushes toward Compliant.

For any specific proposal, the contribution of each word is:
`contribution = tfidf_score x weight`

Where `tfidf_score` is how prominently that word appears in this specific proposal.
Words that do not appear in the proposal have `tfidf_score = 0`, so their contribution is 0.

**Why does this matter for the LLM?**
Without trigger words, the LLM would have to guess why the classifier flagged this proposal.
With trigger words, the LLM can say "the classifier flagged this because of words like
'automatically', 'without', 'users' -- these are transparency-related terms."
This grounds the LLM's reasoning in real evidence, not guesswork.

**EU AI Act Article 14:**
Human reviewers must understand why a flag was raised.
Trigger words satisfy this requirement: the reviewer sees exactly which words caused the flag.

In [55]:
@tool
def get_trigger_words(text:str, n:int=10)->str:
    """
    Find which words in this specific proposal drove the classifier's prediction.
    Use this to ground your reasoning in the model's actual evidence for this proposal.
    Args:
        text: the proposal text to explain
        n: how many top contributing words to return
    Returns:
        the top words with their contribution scores and direction
    """
    results = explain_prediction(classifier_pipeline, text, n)
    output = ""
    for word, score in results:
        direction = "toward Red Flag" if score > 0 else "toward Compliant"
        output += f"{word}: {round(score, 4)} ({direction})\n"
    return output

# Sanity check
print(get_trigger_words.invoke({
    "text": "Our chatbot will handle customer complaints automatically without telling users it is an AI.",
    "n": 5
}))

it: 0.4499 (toward Red Flag)
without: 0.1494 (toward Red Flag)
our: 0.0855 (toward Red Flag)
users: 0.0804 (toward Red Flag)
automatically: 0.0648 (toward Red Flag)



### Tool 3 -- get_policy_severity (Fixed Lookup)

Severity is NOT something the LLM should reason about or decide.
It is a fixed value defined in the policy catalogue:
- RAI-01 through RAI-05 are always High severity
- RAI-06 through RAI-09 are always Medium severity

If we let the LLM decide severity, it might give different answers on different runs.
A deterministic lookup table guarantees the same answer every single time.

**Why severity matters:**
- High severity violations are routed to a human reviewer before any action is taken.
- Medium severity violations proceed automatically with an optional spot check.

This is the Human-in-the-Loop gate required by EU AI Act Article 14.

In [56]:
@tool
def get_policy_severity(policy_id:str)->str:
    """
    Look up the fixed severity level for a given RAI policy.
    Severity is a fixed value from the policy catalogue, not something to reason about.
    Use this to determine whether a violation needs human approval (High) or can proceed automatically (Medium).
    Args:
        policy_id: the policy identifier, e.g. RAI-01, RAI-02, up to RAI-09
    Returns:
        the severity level, either High or Medium, and what it means for routing
    """
    severity_lookup = {
        "RAI-01": "High", "RAI-02": "High", "RAI-03": "High",
        "RAI-04": "High", "RAI-05": "High", "RAI-06": "Medium",
        "RAI-07": "Medium", "RAI-08": "Medium", "RAI-09": "Medium",
    }
    severity = severity_lookup.get(policy_id.upper(), None)
    if severity is None:
        return f"{policy_id} is not a recognised RAI policy. Valid options are RAI-01 through RAI-09."
    if severity == "High":
        return f"{policy_id} is High severity. This violation must be routed to a human reviewer for approval before any action is taken."
    return f"{policy_id} is Medium severity. This violation may proceed to automatic remediation with an optional spot check."

# Sanity check
print(get_policy_severity.invoke({"policy_id": "RAI-02"}))
print(get_policy_severity.invoke({"policy_id": "RAI-07"}))
print(get_policy_severity.invoke({"policy_id": "RAI-99"}))

RAI-02 is High severity. This violation must be routed to a human reviewer for approval before any action is taken.
RAI-07 is Medium severity. This violation may proceed to automatic remediation with an optional spot check.
RAI-99 is not a recognised RAI policy. Valid options are RAI-01 through RAI-09.


### Tool 4 -- log_decision (EU AI Act Article 12 Audit Trail)

Every compliance decision must be logged in a way that allows reconstruction after the fact.
This is required by EU AI Act Article 12: systems must automatically log events
in a way that enables traceability of the system's functioning.

Our audit trail records:
- What the true label was (actual)
- What the model predicted (predicted)
- How confident the model was (probability)
- Whether the prediction was correct (correct)
- Which words drove the prediction (trigger words)
- Which method was used to explain the prediction (always LogReg coef_ x tfidf)

The explanation method is hardcoded as a string because LogReg coefficients are
exact and deterministic. The same input always produces the same explanation.
This is what makes it legally defensible under Article 12.

In [57]:
@tool
def log_decision(text:str, y_true:int, y_pred:int, proba:float, n_terms:int=10)->str:
    """
    Log a compliance decision for audit purposes.
    Use this as your final action once you have reached a conclusion about a proposal.
    Satisfies EU AI Act Article 12: every decision is logged with the exact inputs,
    prediction, probability, and trigger words that produced it.
    Args:
        text: the proposal text that was assessed
        y_true: the true label, 1 for Red Flag, 0 for Compliant
        y_pred: the predicted label, 1 for Red Flag, 0 for Compliant
        proba: the classifier's predicted probability of Red Flag
        n_terms: how many trigger words to include in the log
    Returns:
        a formatted string of the full audit log entry
    """
    log = audit_trail(classifier_pipeline, text, y_true, y_pred, proba, n_terms)

    output = ""
    output += f"Actual label      : {'Red Flag' if log['actual']==1 else 'Compliant'}\n"
    output += f"Predicted label   : {'Red Flag' if log['predicted']==1 else 'Compliant'}\n"
    output += f"Probability       : {log['probability']}\n"
    output += f"Correct           : {log['correct']}\n"
    output += f"Trigger words     : {', '.join(log['trigger_words'])}\n"
    output += f"Explanation method: {log['explanation_method']}\n"
    return output

# Sanity check
print(log_decision.invoke({
    "text": "Our chatbot will handle customer complaints automatically without telling users it is an AI.",
    "y_true": 1,
    "y_pred": 1,
    "proba": 0.87,
    "n_terms": 5
}))

Actual label      : Red Flag
Predicted label   : Red Flag
Probability       : 0.87
Correct           : True
Trigger words     : it, without, our, users, automatically
Explanation method: LogReg coef_ x tfidf



### Tool 5 -- write_compliance_verdict (Structured Final Output)

This is the last tool the LLM calls after the audit log is written.
It produces a structured, human-readable compliance verdict that answers three questions:
1. Which policy was violated?
2. Why was it flagged (grounded in trigger words)?
3. What must be changed to become compliant (grounded in retrieved policy text)?

Severity is looked up deterministically here too, same reason as Tool 3:
we never let the LLM decide severity by itself.

In [58]:
@tool
def write_compliance_verdict(policy_id:str, reason:str, recommended_fix:str)->str:
    """
    Write the final compliance verdict for a flagged proposal.
    Call this as your last action after log_decision.
    Args:
        policy_id: the policy that was violated, e.g. RAI-02
        reason: one sentence explaining what the proposal is missing, grounded in the trigger words
        recommended_fix: one sentence describing what the proposal must add to become compliant
    Returns:
        a formatted compliance verdict string
    """
    severity_lookup = {
        "RAI-01": "High", "RAI-02": "High", "RAI-03": "High",
        "RAI-04": "High", "RAI-05": "High", "RAI-06": "Medium",
        "RAI-07": "Medium", "RAI-08": "Medium", "RAI-09": "Medium",
    }
    severity = severity_lookup.get(policy_id.upper(), "Unknown")

    output = ""
    output += f"POLICY VIOLATED  : {policy_id.upper()}\n"
    output += f"SEVERITY         : {severity}\n"
    output += f"REASON           : {reason}\n"
    output += f"RECOMMENDED FIX  : {recommended_fix}\n"
    return output

---
## Step 7 -- LLM Setup

### Why Groq instead of a local model?

We originally planned to use `phi4-mini` running locally through Ollama.
phi4-mini is a 2.5GB model. During testing, it could not produce structured tool calls
reliably. It wrote tool names as plain text in the message content instead of using
the structured format LangChain's `ToolNode` requires (`tool_calls` field).

We switched to Groq's free API running `llama-3.1-8b-instant`.
Groq runs the model on their servers so our local RAM (6GB) is not a constraint.
`llama-3.1-8b-instant` handles structured tool calling correctly.

### temperature=0.0

Temperature controls how creative vs deterministic the LLM is.
0.0 means fully deterministic: the same input always produces the same output.
This is essential for a compliance system where reproducibility is required by law (Article 12).

### bind_tools(tools)

This attaches all 5 tools to the LLM permanently.
After this, every call to `llm_with_tools` gives the LLM access to all 5 tools.
The LLM sees each tool's name, docstring, and argument types.
It decides on its own which tools to call and when.

### Security note

Your API key is visible in this notebook. Before sharing the notebook publicly,
replace the key with an environment variable: `import os; os.environ["GROQ_API_KEY"]`

In [60]:
from dotenv import load_dotenv
import os

load_dotenv(r"C:\MyGithubSpace\Data-Ethics\.env")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,
    api_key=os.environ.get("GROQ_API_KEY")
)

tools = [
    search_policies,
    get_trigger_words,
    get_policy_severity,
    log_decision,
    write_compliance_verdict,
]

llm_with_tools = llm.bind_tools(tools)
print("LLM loaded and tools bound.")
print(f"Tools available to LLM: {[t.name for t in tools]}")

LLM loaded and tools bound.
Tools available to LLM: ['search_policies', 'get_trigger_words', 'get_policy_severity', 'log_decision', 'write_compliance_verdict']


---
## Step 8 -- LangGraph Agent

### What is LangGraph?

LangGraph is a library for building agents as graphs.
A graph has two things: **nodes** (steps that do work) and **edges** (connections between steps).

Think of it like a flowchart. Each box in the flowchart is a node.
Each arrow between boxes is an edge.
LangGraph runs the graph by following the arrows from one node to the next.

### Our graph has 4 nodes

| Node | What it does |
|------|-------------|
| `classify` | Runs Signal 1 (the ML classifier). Always the first node. |
| `agent` | Calls the LLM with tools. Only runs if classifier said Red Flag. |
| `tools` | Executes whichever tool the LLM requested. Then sends result back to `agent`. |
| `human_gate` | Pauses for human approval on High severity. Auto-approves Medium. |

### Our graph has these edges (routing decisions)

```
START --> classify
classify --> agent       (if Red Flag)
classify --> END         (if Compliant, stop here)
agent --> tools          (if LLM requested a tool)
agent --> human_gate     (if LLM is done reasoning)
tools --> agent          (always loop back so LLM reads the tool result)
human_gate --> END       (always stop after human decision)
```

### What is AgentState?

AgentState is the shared memory of the graph.
Every node reads from it and writes back to it.
It is defined as a TypedDict (a dictionary with fixed, named keys and declared types).

`messages: Annotated[list, operator.add]` is the special one:
instead of overwriting the message list, LangGraph APPENDS new messages to it.
This keeps the full conversation history intact across multiple tool calls.

### Step 8a -- LangGraph imports (already done in Step 1)

All LangGraph imports were included in Step 1 above.
This is just a reminder of what each one does:

- `StateGraph` -- the main class for building the graph
- `START` / `END` -- built-in markers for the beginning and end of the graph
- `ToolNode` -- prebuilt node that runs tool calls automatically
- `MemorySaver` -- saves graph state in memory so the human gate can pause and resume
- `HumanMessage` / `SystemMessage` -- message types the LLM understands
- `TypedDict` -- lets us define a dictionary with fixed key names and types
- `Annotated` + `operator.add` -- tells LangGraph to append messages instead of overwrite

### Step 8b -- AgentState (Shared Memory)

In [61]:
class AgentState(TypedDict):
    # The raw proposal text being assessed. Set at the start, never changes.
    proposal_text: str
    # The classifier's hard prediction. 1 = Red Flag, 0 = Compliant.
    y_pred: int
    # The classifier's Red Flag probability score. e.g. 0.6427
    proba: float
    # High or Medium. Written by the LLM after calling get_policy_severity.
    severity: str
    # Full conversation history. Annotated means messages are APPENDED not overwritten.
    messages: Annotated[list, operator.add]
    # True if human approved or Medium severity auto-approved. False if rejected.
    human_approved: bool
    # Plain English summary of the final outcome.
    final_decision: str

print("AgentState defined.")

AgentState defined.


### Step 8c -- Node 1: classify

This node runs Signal 1: the ML classifier.
It always runs first, on every proposal, regardless of what the proposal says.
It is fast (under 1 second) and never calls the LLM.

If it predicts 0 (Compliant), the graph stops immediately.
If it predicts 1 (Red Flag), the graph continues to the LLM agent.

This is the gate that makes the system efficient: the slow, expensive LLM
only runs on proposals that the fast, cheap classifier already suspects.

In [62]:
def classify(state:AgentState)->dict:
    # Read the proposal text from shared state
    text = state["proposal_text"]

    # predict_proba returns shape (1, 2): [[prob_compliant, prob_red_flag]]
    # [0][1] gets the Red Flag probability for the first (only) document
    proba = classifier_pipeline.predict_proba([text])[0][1]

    # predict() applies the threshold (0.4) internally via ThresholdAdjustor
    # Returns 1 for Red Flag, 0 for Compliant
    y_pred = int(classifier_pipeline.predict([text])[0])

    # Return only the fields this node changes.
    # LangGraph merges this dict into AgentState automatically.
    return {
        "y_pred": y_pred,
        "proba": round(float(proba), 4),
    }

print("classify node defined.")

classify node defined.


### Step 8c -- Node 2: agent

This node calls the LLM with all 5 tools available.
It runs in a loop with the `tools` node until the LLM decides it is done.

**The system prompt:**
The system prompt is the instruction we give the LLM about its job.
It is deliberately thin: no policy text, no trigger words, no pre-fetched context.
All of that arrives through tool calls the LLM makes itself.

We embed the exact proposal text and probability directly into the prompt
so the LLM passes the correct values to the tools, not summaries or paraphrases.

**The tool call loop:**
After each LLM response, `route_after_agent` checks if the LLM requested a tool.
If yes, `ToolNode` runs the tool and sends the result back to this node.
The LLM reads the result and decides what to do next.
This continues until the LLM produces a response with no tool calls.

In [ ]:
def agent(state:AgentState)->dict:
    system=SystemMessage(content="""You are a compliance assessment agent for a Responsible AI policy system.

You have access to these tools:
- {get_trigger_words}: find which words in the proposal drove the classifier prediction
- {search_policies}: search the RAI policy catalogue for relevant clauses
- {get_policy_severity}: look up the severity of a specific RAI policy
- {log_decision}: write the audit trail entry for this decision
- {write_compliance_verdict}: write the final structured compliance verdict

Use your tools in whatever order and however many times you judge necessary to accurately assess whether this proposal violates any RAI policy. When you are confident in your assessment, call {log_decision} and then {write_compliance_verdict} to conclude.""")

    messages=[system]+state["messages"]
    response=llm_with_tools.invoke(messages)
    return{"messages":[response]}

### Step 8c -- Node 3: human_gate

This node implements the Human-in-the-Loop gate required by EU AI Act Article 14.

**Medium severity:** auto-approved. The system proceeds without human input.

**High severity:** the node prints the full agent reasoning (every tool call and result)
and waits for a human to type APPROVE or REJECT.

`input()` is a built-in Python function that pauses execution and waits for keyboard input.
The graph does not continue until the human responds.

This is the critical difference between a fully automated system and one with human oversight:
a human can review, understand, and reject any High severity decision before it takes effect.

In [64]:
def human_gate(state:AgentState)->dict:
    severity=state["severity"]

    if severity=="Medium":
        print("Medium severity: proceeding automatically.")
        return{
            "human_approved":True,
            "final_decision":"Medium severity violation. Automatic remediation approved."
        }

    print("\n--- HIGH SEVERITY: HUMAN APPROVAL REQUIRED ---")
    print(f"Proposal text:\n{state['proposal_text']}\n")
    print("Agent reasoning and tool outputs:")
    for message in state["messages"]:
        if hasattr(message, "name") and message.name and message.content:
            print(f"[{message.name}] {message.content}")
        elif message.content and not hasattr(message, "tool_calls"):
            print(f"[LLM] {message.content}")
    print("----------------------------------------------")

    decision=input("Type APPROVE or REJECT: ").strip().upper()

    if decision=="APPROVE":
        return{
            "human_approved":True,
            "final_decision":"High severity violation. Human reviewer approved remediation."
        }

    return{
        "human_approved":False,
        "final_decision":"High severity violation. Human reviewer rejected remediation. No changes made."
    }

### Step 8d -- Routing Functions

Routing functions are not nodes. They are plain Python functions that LangGraph
calls after a node finishes to decide which node runs next.

They always receive the current `state` and return a string: the name of the next node.
Returning `END` tells LangGraph to stop the graph entirely.

**route_after_classify:**
If `y_pred == 1` (Red Flag), go to `agent`.
If `y_pred == 0` (Compliant), stop. No LLM call needed.

**route_after_agent:**
Checks the last message the LLM produced.
If it contains tool calls (`tool_calls` field is non-empty), go to `tools`.
If no tool calls (LLM is done reasoning), go to `human_gate`.

In [65]:
def route_after_classify(state:AgentState)->str:
    if state["y_pred"] == 1:
        return "agent"
    return END

def route_after_agent(state:AgentState)->str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and len(last_message.tool_calls) > 0:
        return "tools"
    return "human_gate"

print("Routing functions defined.")

Routing functions defined.


### Step 8e -- Build and Compile the Graph

In [66]:
# Step 1: Create an empty graph with our state definition
graph_builder = StateGraph(AgentState)

# Step 2: Register nodes
graph_builder.add_node("classify", classify)
graph_builder.add_node("agent", agent)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_node("human_gate", human_gate)

# Step 3: Add edges
# Fixed edge: always start at classify
graph_builder.add_edge(START, "classify")

# Conditional edge: after classify, route based on y_pred
graph_builder.add_conditional_edges(
    "classify",
    route_after_classify,
    {"agent": "agent", END: END}
)

# Conditional edge: after agent, route based on whether LLM called a tool
graph_builder.add_conditional_edges(
    "agent",
    route_after_agent,
    {"tools": "tools", "human_gate": "human_gate"}
)

# Fixed edge: after tools, always go back to agent
graph_builder.add_edge("tools", "agent")

# Fixed edge: after human_gate, always stop
graph_builder.add_edge("human_gate", END)

# Step 4: Compile with memory so state persists across the human gate pause
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print("Graph compiled successfully.")

Graph compiled successfully.


---
## Step 8f -- Run the Agent

### How to use this cell

1. Change `proposal_text` to the proposal you want to assess.
2. Change `thread_id` to a new unique string every time you run (e.g. "test_run_1", "test_run_2").
   If you reuse the same thread_id, LangGraph will try to resume the old run instead of starting fresh.
3. Run the cell.
4. If the proposal is High severity, type APPROVE or REJECT when prompted.

### What the initial_state means

All fields except `proposal_text` and `messages` start at empty/zero defaults.
LangGraph fills them in as each node runs.
`messages` starts with one `HumanMessage` containing the proposal text so the LLM
sees it as the thing it needs to assess.

In [67]:
# Change thread_id every time you run a new proposal
config = {"configurable": {"thread_id": "test_run_12"}}

initial_state = {
    "proposal_text": "Our chatbot will handle customer complaints automatically without telling users it is an AI.",
    "y_pred": 0,
    "proba": 0.0,
    "severity": "",
    "messages": [HumanMessage(content="Our chatbot will handle customer complaints automatically without telling users it is an AI.")],
    "human_approved": False,
    "final_decision": "",
}

result = graph.invoke(initial_state, config=config)

print("\n--- FINAL DECISION ---")
print(result["final_decision"])
print(f"Classifier prediction : {'Red Flag' if result['y_pred']==1 else 'Compliant'}")
print(f"Probability           : {result['proba']}")
print(f"Human approved        : {result['human_approved']}")

print("\n--- COMPLIANCE VERDICT ---")
for message in reversed(result["messages"]):
    if hasattr(message, "name") and message.name == "write_compliance_verdict":
        print(message.content)
        break


--- HIGH SEVERITY: HUMAN APPROVAL REQUIRED ---
Proposal text:
Our chatbot will handle customer complaints automatically without telling users it is an AI.

Agent reasoning and tool outputs:
[LLM] Our chatbot will handle customer complaints automatically without telling users it is an AI.
[get_trigger_words] it: 0.4499 (toward Red Flag)
without: 0.1494 (toward Red Flag)
our: 0.0855 (toward Red Flag)
users: 0.0804 (toward Red Flag)
automatically: 0.0648 (toward Red Flag)
ai: 0.0609 (toward Red Flag)
users it: 0.0451 (toward Red Flag)
automatically without: 0.036 (toward Red Flag)
handle: -0.0258 (toward Compliant)
complaints: -0.0239 (toward Compliant)

[search_policies] [page 2]
Violation Patterns
A proposal describes a chatbot, virtual assistant, or automated messaging system with no mention of
disclosing to the user that they are talking to an AI system. A proposal describes an automated
decision, such as an approval, denial, or score, being delivered to a customer with no statement 